# Laboratorio 4 (Parte 2) — Inciso 4: Construcción de modelos de Machine Learning

Se construyen los tres modelos mínimos requeridos (Regresión Logística, Random Forest y Gradient Boosting), se entrenan con una división convencional 70/30, y se ajustan sus hiperparámetros mediante búsqueda aleatoria. El conjunto de prueba (30%) se mantiene fijo para todos los modelos y se reutiliza en el inciso 5.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import RUTA_DATA_PROCESSED
from src.modelado import (
    GRIDS_HIPERPARAMETROS,
    PREDICTORES,
    agregar_features,
    ajustar_hiperparametros,
    construir_modelos_base,
    construir_respuesta,
    dividir_datos,
    entrenar_modelos,
    guardar_modelo,
)

muestra = pd.read_parquet(RUTA_DATA_PROCESSED / "dataset_ml_muestra.parquet")
muestra = construir_respuesta(agregar_features(muestra))
print(f"{len(muestra):,} observaciones en la muestra de trabajo, {len(PREDICTORES)} predictoras")

299,992 observaciones en la muestra de trabajo, 11 predictoras


## 4.1 Modelos base

Se instancian los tres modelos mínimos requeridos (`src.modelado.construir_modelos_base`): Regresión Logística (con escalado en un `Pipeline`, por ser sensible a la escala de los predictores), Random Forest y Gradient Boosting. Regresión Logística y Random Forest reciben `class_weight="balanced"` para compensar el desbalance de clases del inciso 2.4; `GradientBoostingClassifier` no admite ese parámetro, así que su desbalance se compensa con `sample_weight` durante el entrenamiento.

In [2]:
modelos_base = construir_modelos_base()
for nombre, modelo in modelos_base.items():
    print(f"{nombre}: {type(modelo).__name__ if not hasattr(modelo, 'steps') else 'Pipeline(' + ', '.join(s[0] for s in modelo.steps) + ')'}")

regresion_logistica: Pipeline(escalado, modelo)
random_forest: RandomForestClassifier
gradient_boosting: GradientBoostingClassifier


## 4.2 División 70/30 y entrenamiento inicial

`dividir_datos` aplica una división estratificada 70/30 sobre `alta_cianobacteria`, con semilla fija (`SEMILLA = 42`), de modo que el mismo conjunto de prueba se reproduce en cualquier notebook que lo vuelva a calcular sobre esta misma muestra (inciso 4.4).

In [3]:
X_train, X_test, y_train, y_test = dividir_datos(muestra)
print(f"Entrenamiento: {len(X_train):,} ({y_train.mean():.1%} clase 1)")
print(f"Prueba:        {len(X_test):,} ({y_test.mean():.1%} clase 1)")

modelos_iniciales = entrenar_modelos(construir_modelos_base(), X_train, y_train)
print("Modelos entrenados con hiperparámetros por defecto:", list(modelos_iniciales))

Entrenamiento: 209,994 (5.5% clase 1)
Prueba:        89,998 (5.5% clase 1)


Modelos entrenados con hiperparámetros por defecto: ['regresion_logistica', 'random_forest', 'gradient_boosting']


## 4.3 Ajuste de hiperparámetros

Para cada modelo se define un grid pequeño centrado en los hiperparámetros que más afectan su balance sesgo/varianza (`src.modelado.GRIDS_HIPERPARAMETROS`):

- **Regresión Logística**: `C` (inverso de la regularización), en `[0.01, 0.1, 1, 10]`.
- **Random Forest**: `n_estimators`, `max_depth` y `min_samples_leaf`, que controlan el tamaño del bosque y qué tan profundo/específico puede volverse cada árbol.
- **Gradient Boosting**: `n_estimators`, `learning_rate` y `max_depth`, que controlan cuántas etapas de boosting se suman y qué tan agresiva es cada una.

La búsqueda usa `RandomizedSearchCV` con validación cruzada de 3 particiones sobre el conjunto de entrenamiento, optimizando **ROC-AUC**: es la métrica menos sensible al desbalance de clases del inciso 2.4, a diferencia de accuracy, y no exige fijar de antemano un umbral de decisión como sí lo requieren precision/recall/F1.

In [4]:
modelos_ajustados = {}
mejores_parametros = {}
for nombre, modelo in construir_modelos_base().items():
    mejor, params = ajustar_hiperparametros(nombre, modelo, X_train, y_train, n_iter=8, cv=3)
    modelos_ajustados[nombre] = mejor
    mejores_parametros[nombre] = params
    print(f"{nombre}: {params}")

tabla_hiperparametros = pd.DataFrame(mejores_parametros).T
tabla_hiperparametros.to_csv(RUTA_DATA_PROCESSED / "p2_hiperparametros.csv")
tabla_hiperparametros

/home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/DATA_SCIENCE/venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:326: UserWarning: The total space of parameters 4 is smaller than n_iter=8. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


regresion_logistica: {'modelo__C': 10.0}


random_forest: {'n_estimators': 400, 'min_samples_leaf': 5, 'max_depth': None}


gradient_boosting: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1}


,modelo__C,n_estimators,min_samples_leaf,max_depth,learning_rate
regresion_logistica,10.0,NaN,NaN,NaN,NaN
random_forest,NaN,400.0,5.0,NaN,NaN
gradient_boosting,NaN,100.0,NaN,4.0,0.1


## 4.4 Conjunto de prueba compartido

`X_test`/`y_test` se generan una sola vez, a partir de la misma llamada a `dividir_datos` con semilla fija, y se usan para evaluar los tres modelos en el inciso 5, garantizando una comparación justa. Se guardan los tres modelos ajustados en `data/processed/modelos/` para reutilizarlos en los incisos 5, 7, 8 y 9 sin reentrenar.

In [5]:
for nombre, modelo in modelos_ajustados.items():
    guardar_modelo(nombre, modelo)

X_test.assign(alta_cianobacteria=y_test).to_parquet(RUTA_DATA_PROCESSED / "p2_test_set.parquet", index=False)
print("Modelos guardados en data/processed/modelos/, conjunto de prueba guardado en p2_test_set.parquet")

Modelos guardados en data/processed/modelos/, conjunto de prueba guardado en p2_test_set.parquet


### Self-check

In [6]:
assert set(modelos_ajustados) == {"regresion_logistica", "random_forest", "gradient_boosting"}
assert len(X_train) + len(X_test) == len(muestra)
assert abs(y_train.mean() - y_test.mean()) < 0.01, "el split estratificado no preservó la proporción de clases"
for nombre, params in mejores_parametros.items():
    assert set(params).issubset(GRIDS_HIPERPARAMETROS[nombre]), f"{nombre}: hiperparámetro fuera del grid definido"
print("OK: 3 modelos ajustados, split estratificado consistente, hiperparámetros dentro de los grids definidos.")

OK: 3 modelos ajustados, split estratificado consistente, hiperparámetros dentro de los grids definidos.
